In [ ]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


#### FIX ME #####
# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################
# FIX ME update with your username and password and CRUD Python module name

username = "aacuser"
password = "Jdshidgcf14fkygsk"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

#Add in Grazioso Salvare’s logo
image_filename = 'Grazioso Salvare Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

#FIX ME Place the HTML image tag in the line below into the app.layout code according to your design
#FIX ME Also remember to include a unique identifier such as your name or date
#html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()))

app.layout = html.Div([
    html.Center([
        html.Img(
            src='data:image/png;base64,{}'.format(encoded_image.decode()),
            style={'height': '120px'}
        ),
        html.H1('Grazioso Salvare Rescue Dashboard'),
        html.H3('Brandon Fluegge - CS 340 Project Two')
    ]),

    html.Hr(),

    dcc.RadioItems(
        id='filter-type',
        options=[
            {'label': 'Water Rescue', 'value': 'water'},
            {'label': 'Mountain or Wilderness Rescue', 'value': 'mountain'},
            {'label': 'Disaster or Individual Tracking', 'value': 'disaster'},
            {'label': 'Reset', 'value': 'reset'}
        ],
        value='reset',
        labelStyle={'display': 'inline-block', 'margin-right': '20px'}
    ),

    html.Hr(),

    dash_table.DataTable(
        id='datatable-id',
        columns=[
            {
                "name": i,
                "id": i,
                "deletable": False,
                "selectable": True
            }
            for i in df.columns
        ],
        data=df.to_dict('records'),

        page_size=10,
        sort_action='native',
        filter_action='native',
        row_selectable='single',
        selected_rows=[],
        selected_columns=[],

        style_table={
            'overflowX': 'auto'
        },

        style_cell={
            'textAlign': 'left',
            'minWidth': '100px',
            'maxWidth': '250px',
            'whiteSpace': 'normal'
        }
    ),

    html.Br(),
    html.Hr(),

    html.Div(
        className='row',
        style={'display': 'flex'},
        children=[
            html.Div(
                id='graph-id',
                className='col s12 m6',
                style={'width': '50%'}
            ),

            html.Div(
                id='map-id',
                className='col s12 m6',
                style={'width': '50%'}
            )
        ]
    )
])

#############################################
# Interaction Between Components / Controller
#############################################
    
@app.callback(
    Output('datatable-id', 'data'),
    [Input('filter-type', 'value')]
)
def update_dashboard(filter_type):

    if filter_type == 'water':
        query = {
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "Labrador Retriever Mix",
                    "Chesapeake Bay Retriever",
                    "Newfoundland"
                ]
            },
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {
                "$gte": 26,
                "$lte": 156
            }
        }

    elif filter_type == 'mountain':
        query = {
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "German Shepherd",
                    "Alaskan Malamute",
                    "Old English Sheepdog",
                    "Siberian Husky",
                    "Rottweiler"
                ]
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {
                "$gte": 26,
                "$lte": 156
            }
        }

    elif filter_type == 'disaster':
        query = {
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "Doberman Pinscher",
                    "German Shepherd",
                    "Golden Retriever",
                    "Bloodhound",
                    "Rottweiler"
                ]
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {
                "$gte": 20,
                "$lte": 300
            }
        }

    else:
        # Reset dashboard to the complete data set
        query = {}

    filtered_df = pd.DataFrame.from_records(db.read(query))

    # MongoDB ObjectIds cannot be serialized by the Dash DataTable
    if '_id' in filtered_df.columns:
        filtered_df.drop(columns=['_id'], inplace=True)

    return filtered_df.to_dict('records')


@app.callback(
    Output('graph-id', 'children'),
    [Input('datatable-id', 'derived_virtual_data')]
)
def update_graphs(viewData):

    # Use the complete dataframe if Dash has not generated
    # derived table data yet
    if viewData is None:
        dff = df.copy()
    else:
        dff = pd.DataFrame.from_dict(viewData)

    # Prevent chart errors when a filter returns no records
    if dff.empty or 'breed' not in dff.columns:
        return html.Div("No breed data available.")

    # Count animals by breed
    breed_counts = (
        dff['breed']
        .value_counts()
        .reset_index()
    )

    breed_counts.columns = ['breed', 'count']

    # Limit the pie chart to the most common breeds so it stays readable
    breed_counts = breed_counts.head(10)

    return dcc.Graph(
        figure=px.pie(
            breed_counts,
            names='breed',
            values='count',
            title='Animal Breeds for Selected Rescue Type'
        )
    )
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', 'children'),
    [
        Input('datatable-id', 'derived_virtual_data'),
        Input('datatable-id', 'derived_virtual_selected_rows')
    ]
)
def update_map(viewData, index):

    # Use complete dataframe until Dash provides virtual table data
    if viewData is None:
        dff = df.copy()
    else:
        dff = pd.DataFrame.from_dict(viewData)

    # Nothing to display
    if dff.empty:
        return html.Div("No location data available.")

    # Use first row if user has not selected one
    if index is None or len(index) == 0:
        row = 0
    else:
        row = index[0]

    # Protect against invalid row indexes after filtering
    if row >= len(dff):
        row = 0

    # Get values by COLUMN NAME instead of fragile column numbers
    latitude = dff.iloc[row]['location_lat']
    longitude = dff.iloc[row]['location_long']
    breed = dff.iloc[row]['breed']
    animal_name = dff.iloc[row]['name']

    # Handle missing coordinates
    if pd.isna(latitude) or pd.isna(longitude):
        return html.Div("Location unavailable for selected animal.")

    # Handle unnamed animals
    if pd.isna(animal_name):
        animal_name = "Unnamed Animal"

    return dl.Map(
        style={
            'width': '100%',
            'height': '500px'
        },
        center=[latitude, longitude],
        zoom=10,
        children=[
            dl.TileLayer(),

            dl.Marker(
                position=[latitude, longitude],
                children=[
                    dl.Tooltip(str(breed)),
                    dl.Popup([
                        html.H3("Animal Name"),
                        html.P(str(animal_name)),
                        html.P("Breed: " + str(breed))
                    ])
                ]
            )
        ]
    )


# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server() 

/home/codio/.pyenv/versions/3.11.2/lib/python3.11/site-packages/jupyter_dash/comms.py:100: RuntimeWarning: coroutine 'Kernel.execute_request' was never awaited
  kernel.execute_request(stream, ident, parent)


Dash app running on https://typerapid-henryfame-3000.codio.io/proxy/8050/
